In [ ]:
model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [10]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    validation_data=(X_test, y_test),
    verbose=1
)

644/644 ━━━━━━━━━━━━━━━━━━━━ 27s 42ms/step - accuracy: 0.1699 - loss: 4.1318 - val_accuracy: 0.0532 - val_loss: 8.4648
Epoch 22/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 24s 37ms/step - accuracy: 0.1836 - loss: 4.0380 - val_accuracy: 0.0567 - val_loss: 8.6038
Epoch 23/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 23s 35ms/step - accuracy: 0.1980 - loss: 3.9431 - val_accuracy: 0.0565 - val_loss: 8.7515
Epoch 24/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.2147 - loss: 3.8559 - val_accuracy: 0.0567 - val_loss: 8.8413
Epoch 25/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - accuracy: 0.2280 - loss: 3.7737 - val_accuracy: 0.0565 - val_loss: 8.9844
Epoch 26/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.2399 - loss: 3.6928 - val_accuracy: 0.0567 - val_loss: 9.1132
Epoch 27/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 19s 30ms/step - accuracy: 0.2532 - loss: 3.6145 - val_accuracy: 0.0556 - val_loss: 9.2023
Epoch 28/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.2665 - loss: 3.5362 - val_

In [15]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences


def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]

    if len(token_list) >= max_sequence_len:
        token_list = token_list[-(max_sequence_len - 1):]

    token_list = pad_sequences(
        [token_list],
        maxlen=max_sequence_len - 1,
        padding="pre"
    )

    predicted_probabilities = model.predict(token_list, verbose=0)
    predicted_index = np.argmax(predicted_probabilities, axis=1)[0]

    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            return word

    return None

In [16]:
input_text = "to be or not to be"

max_sequence_len = model.input_shape[1] + 1

next_word = predict_next_word(
    model, tokenizer, input_text, max_sequence_len
)

print("Input text:", input_text)
print("Predicted next word:", next_word)

Input text: to be or not to be
Predicted next word: buried


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input

model = Sequential([
    Input(shape=(max_sequence_len - 1,)),
    Embedding(input_dim=total_words, output_dim=100),
    LSTM(150, return_sequences=True),
    Dropout(0.2),
    LSTM(100),
    Dense(total_words, activation="softmax")
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 13, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 13, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 13, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,219,418 (4.65 MB)

 Trainable params: 1,219,418 (4.65 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
model.save("next_word_lstm.h5")

In [18]:
import pickle

with open("tokenizer.pickle", "wb") as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [19]:
model.save("next_word_lstm.h5")

In [20]:
model.save("next_word_lstm.keras")